In [1]:
# Standard library
from pathlib import Path
import random
import time


# Numerical computing
import numpy as np
import pandas as pd


# Visualization
import matplotlib.pyplot as plt


# PyTorch
import torch
import torch.nn as nn
import torch.optim as optim


# Computer vision
import torchvision
from torchvision import datasets, transforms, models


# Data loading
from torch.utils.data import DataLoader


# Evaluation metrics
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)


print("Environment initialized")
print(f"PyTorch     : {torch.__version__}")
print(f"Torchvision : {torchvision.__version__}")

Environment initialized
PyTorch     : 2.12.0+cpu
Torchvision : 0.27.0+cpu


In [2]:
# Reproducibility

SEED = 42


random.seed(SEED)

np.random.seed(SEED)

torch.manual_seed(SEED)

torch.cuda.manual_seed_all(SEED)



# Device configuration

device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)


print(f"Using device: {device}")

Using device: cpu


In [4]:
# Dataset paths

datasetRoot = Path("../dataset")


trainPath = datasetRoot / "train"
valPath = datasetRoot / "val"
testPath = datasetRoot / "test"


# Transforms


trainTransforms = transforms.Compose(
    [
        transforms.Resize((224,224)),

        transforms.RandomHorizontalFlip(),

        transforms.RandomRotation(10),

        transforms.ToTensor(),

        transforms.Normalize(
            mean=[0.485,0.456,0.406],
            std=[0.229,0.224,0.225]
        )
    ]
)



evalTransforms = transforms.Compose(
    [
        transforms.Resize((224,224)),

        transforms.ToTensor(),

        transforms.Normalize(
            mean=[0.485,0.456,0.406],
            std=[0.229,0.224,0.225]
        )
    ]
)


# Datasets


trainDataset = datasets.ImageFolder(
    trainPath,
    transform=trainTransforms
)


valDataset = datasets.ImageFolder(
    valPath,
    transform=evalTransforms
)


testDataset = datasets.ImageFolder(
    testPath,
    transform=evalTransforms
)



classNames = trainDataset.classes
numClasses = len(classNames)


print(classNames)

['AMD', 'CNV', 'CSR', 'DME', 'DR', 'DRUSEN', 'MH', 'NORMAL']


In [5]:
batchSize = 32


trainLoader = DataLoader(
    trainDataset,
    batch_size=batchSize,
    shuffle=True
)


valLoader = DataLoader(
    valDataset,
    batch_size=batchSize,
    shuffle=False
)


testLoader = DataLoader(
    testDataset,
    batch_size=batchSize,
    shuffle=False
)



print("DataLoaders ready")

DataLoaders ready


In [7]:
# Load pretrained ResNet50 model

model = models.resnet50(
    weights=models.ResNet50_Weights.IMAGENET1K_V2
)


# Replace ImageNet classifier with OCT classifier

inputFeatures = model.fc.in_features

model.fc = nn.Linear(
    inputFeatures,
    numClasses
)


# Move model to GPU/CPU

model = model.to(device)


print("ResNet50 initialized")
print(model.fc)

ResNet50 initialized
Linear(in_features=2048, out_features=8, bias=True)


In [8]:
criterion = nn.CrossEntropyLoss()


optimizer = optim.Adam(
    model.parameters(),
    lr=0.0001
)


print("Training setup complete")

Training setup complete


In [9]:
images, labels = next(iter(trainLoader))

images = images.to(device)


outputs = model(images)


print(outputs.shape)

torch.Size([32, 8])


In [10]:
def trainOneEpoch(model, dataLoader, criterion, optimizer, device):

    # Put model in training mode
    model.train()


    runningLoss = 0.0
    correctPredictions = 0
    totalSamples = 0


    for images, labels in dataLoader:

        # Move batch to device
        images = images.to(device)
        labels = labels.to(device)


        # Clear old gradients
        optimizer.zero_grad()


        # Forward pass
        outputs = model(images)


        # Calculate loss
        loss = criterion(outputs, labels)


        # Backpropagation
        loss.backward()


        # Update weights
        optimizer.step()


        # Track loss
        runningLoss += loss.item() * images.size(0)


        # Convert logits → predictions
        _, predictions = torch.max(outputs, 1)


        # Track accuracy
        correctPredictions += (
            predictions == labels
        ).sum().item()


        totalSamples += labels.size(0)



    epochLoss = runningLoss / totalSamples

    epochAccuracy = (
        correctPredictions / totalSamples
    )


    return epochLoss, epochAccuracy

In [11]:
def validateModel(model, dataLoader, criterion, device):

    # Put model in evaluation mode
    model.eval()


    runningLoss = 0.0
    correctPredictions = 0
    totalSamples = 0


    # Disable gradient calculations
    with torch.no_grad():

        for images, labels in dataLoader:


            # Move batch to device
            images = images.to(device)
            labels = labels.to(device)


            # Forward pass
            outputs = model(images)


            # Calculate loss
            loss = criterion(outputs, labels)


            # Track loss
            runningLoss += loss.item() * images.size(0)


            # Convert logits to predictions
            _, predictions = torch.max(outputs, 1)


            # Track accuracy
            correctPredictions += (
                predictions == labels
            ).sum().item()


            totalSamples += labels.size(0)



    epochLoss = runningLoss / totalSamples


    epochAccuracy = (
        correctPredictions / totalSamples
    )


    return epochLoss, epochAccuracy

In [13]:
# Training configuration

numEpochs = 5


history = {
    "trainLoss": [],
    "trainAccuracy": [],
    "valLoss": [],
    "valAccuracy": []
}


bestValAccuracy = 0.0


# Start training

for epoch in range(numEpochs):

    startTime = time.time()


    # Training phase

    trainLoss, trainAccuracy = trainOneEpoch(
        model,
        trainLoader,
        criterion,
        optimizer,
        device
    )


    # Validation phase

    valLoss, valAccuracy = validateModel(
        model,
        valLoader,
        criterion,
        device
    )


    # Store results

    history["trainLoss"].append(trainLoss)
    history["trainAccuracy"].append(trainAccuracy)

    history["valLoss"].append(valLoss)
    history["valAccuracy"].append(valAccuracy)



    # Save best model

    if valAccuracy > bestValAccuracy:

        bestValAccuracy = valAccuracy

        torch.save(
            model.state_dict(),
            "../models/bestResNet50.pth"
        )


    epochTime = time.time() - startTime



    print(
        f"Epoch [{epoch+1}/{numEpochs}] "
        f"| "
        f"Train Loss: {trainLoss:.4f} "
        f"| "
        f"Train Acc: {trainAccuracy:.4f} "
        f"| "
        f"Val Loss: {valLoss:.4f} "
        f"| "
        f"Val Acc: {valAccuracy:.4f} "
        f"| "
        f"Time: {epochTime:.1f}s"
    )


print("\nTraining complete")
print(f"Best validation accuracy: {bestValAccuracy:.4f}")

Epoch [1/5] | Train Loss: 0.3119 | Train Acc: 0.8997 | Val Loss: 0.1267 | Val Acc: 0.9561 | Time: 6241.3s


KeyboardInterrupt: 